In [27]:
import os
import import_ipynb
import requests
import pandas as pd
import numpy as np

from glob import glob
from typing import Optional
from collections import defaultdict
from datetime import datetime, timedelta
from face_cf import APICallNode, run_face, real_time_override_check_with_face

In [ ]:
url = "your-url"
API_KEY  = "your-key"

HOUSE_ID = 29104                                
ZONE     = 1 

new_data = pd.read_csv("eda/full_data.csv")

In [29]:
# Merge room temperature to events data
data_path = "/Users/op24226/Desktop/Passiv/Data/PSTData5"

settings_files = sorted(glob(os.path.join(data_path, "*.csv")))

def get_house_id(filepath: str) -> str:
    return os.path.basename(filepath).split("_")[0]

# Load into a dict: {house_id: dataframe}
temp_by_house = defaultdict(list)

for f in settings_files:
    hid = get_house_id(f)
    df = pd.read_csv(f)
    df["house_id"] = hid
    temp_by_house[hid].append(df)

temp_by_house = {hid: pd.concat(dfs, ignore_index=True) for hid, dfs in temp_by_house.items()}

house_id = "29104"   
room_temp_df = temp_by_house[house_id]

# Parse timestamps
room_temp_df["Time (UTC)"] = pd.to_datetime(room_temp_df['Time (UTC)'], utc=True)
new_data["Timestamp"] = pd.to_datetime(new_data["Timestamp"], utc=True)

room_temp_df = room_temp_df.dropna(subset=['Time (UTC)'])

# Sort both by timestamp (required for merge_asof)
room_temp_df = room_temp_df.sort_values('Time (UTC)')
events_df = new_data.sort_values("Timestamp")

# Merge: for each event, find the closest prior room temp reading
merged = pd.merge_asof(
    events_df,
    room_temp_df[["Time (UTC)", "Room temperature (Zone 1) (°C)", "Room temperature (Zone 2) (°C)", 'External temperature (°C)', 'Tariff rate (p/kWh)']],
    left_on="Timestamp",
    right_on="Time (UTC)",
    direction="backward"
)

In [30]:
def prepare_new_data(new_data: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans up new_data so it's ready to extract nodes from.

    INPUT:  raw new_data DataFrame
    OUTPUT: cleaned DataFrame with UTC timestamps
    """
    df = new_data.copy()

    # Ensure Timestamp is UTC-aware
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    if df["Timestamp"].dt.tz is None:
        df["Timestamp"] = df["Timestamp"].dt.tz_localize("UTC")
    else:
        df["Timestamp"] = df["Timestamp"].dt.tz_convert("UTC")

    # Ensure override_end is UTC-aware
    df["override_end"] = pd.to_datetime(df["override_end"], errors="coerce")
    if df["override_end"].notna().any():
        mask = df["override_end"].notna()
        if df.loc[mask, "override_end"].dt.tz is None:
            df.loc[mask, "override_end"] = df.loc[mask, "override_end"].dt.tz_localize("UTC")
        else:
            df.loc[mask, "override_end"] = df.loc[mask, "override_end"].dt.tz_convert("UTC")

    df["zone_int"] = pd.array(
        df["zone"].apply(
            lambda z: int(float(z)) if pd.notna(z) else pd.NA
        ),
        dtype="Int64"
    )

    return df

In [31]:
def extract_schedule_for_event(
    df: pd.DataFrame,
    house_id: int,
    zone: int,
    event_ts: pd.Timestamp
) -> dict:
    """
    Extracts the deduplicated heating schedule for a given event timestamp.

    The schedule rows at each timestamp contain the 7-day weekly schedule
    repeated as individual rows. We take the unique (time, C) combinations
    to get the actual schedule slots.

    INPUT:
        df       : prepared new_data DataFrame
        house_id : which house
        zone     : which zone (int)
        event_ts : the exact Timestamp of the event

    OUTPUT:
        schedule dict {"hours": [0, 6, 16, 22], "C": [17, 19, 20, 17]}
    """
    zone_mask = df["zone_int"].fillna(-1).astype(int) == int(zone)

    event_df = df[
        (df["house_id"] == house_id) &
        zone_mask &
        (df["Timestamp"] == event_ts) &
        (df["time"].notna()) &
        (df["C"].notna()) &
        (df["override_end"].isna())   # schedule rows only, not override rows
    ].copy()

    if event_df.empty:
        print(f"No schedule found at {event_ts} — using frost default")
        return {"hours": [0], "C": ["frost"]}

    # Convert "HH:MM" to integer hours
    event_df["hour"] = pd.to_datetime(
        event_df["time"], format="%H:%M", errors="coerce"
    ).dt.hour

    # Deduplicate — the schedule is repeated 7x (once per weekday)
    # keep the first occurrence of each (hour, C) pair
    event_df = event_df.dropna(subset=["hour"])
    event_df = event_df.drop_duplicates(subset=["hour", "C"], keep="first")
    event_df = event_df.sort_values("hour")

    hours = event_df["hour"].astype(int).tolist()
    temps = []
    for t in event_df["C"].tolist():
        if str(t).lower() == "frost":
            temps.append("frost")
        else:
            try:
                temps.append(float(t))
            except (ValueError, TypeError):
                temps.append("frost")

    print(f"  Schedule at {event_ts}: hours={hours}, temps={temps}")
    return {"hours": hours, "C": temps}


In [32]:
def extract_override_for_event(
    df: pd.DataFrame,
    house_id: int,
    zone: int,
    event_ts: pd.Timestamp
) -> Optional[dict]:
    """
    Extracts the override event at a given timestamp, if one exists.

    INPUT:
        df       : prepared new_data DataFrame
        house_id : which house
        zone     : which zone (int)
        event_ts : the exact Timestamp of the event

    OUTPUT:
        dict with override_time, override_end, override_temp
        OR None if no override at this timestamp
    """
    zone_mask = df["zone_int"].fillna(-1).astype(int) == int(zone)

    override_df = df[
        (df["house_id"] == house_id) &
        zone_mask &
        (df["Timestamp"] == event_ts) &
        (df["override_end"].notna()) &
        (df["C"].notna())
    ].copy()

    if override_df.empty:
        return None

    # Take the first valid override row
    for _, row in override_df.iterrows():
        try:
            override_temp = float(row["C"])
        except (ValueError, TypeError):
            continue

        override_end = row["override_end"].to_pydatetime()

        # Skip corrupt override_end dates (e.g. 1970-01-01)
        if override_end.year < 2025 or override_end.year > 2030:
            print(f"  Skipping override with suspicious end date: {override_end}")
            continue

        return {
            "override_time": row["Timestamp"].to_pydatetime(),
            "override_end":  override_end,
            "override_temp": override_temp
        }

    return None


In [33]:
def call_api_for_day(
    api_url: str,
    house_id: int,
    zone: int,
    date: datetime.date,
    schedule: dict,
    zone_start_temp: float = 19.0,
    tariff: dict = None
) -> dict:
    """
    Builds optim_inputs from extracted schedule and calls the API.

    INPUT:
        api_url         : your model API URL
        house_id        : which house (for logging only)
        zone            : which zone
        date            : which date (sets start_datetime)
        schedule        : {"hours": [...], "C": [...]} from extract_schedule_for_day
        zone_start_temp : starting room temperature
        tariff          : {"hours": [...], "cost": [...]}
                          if None uses a default flat tariff 

    OUTPUT:
        full JSON response from the model API
        contains real_dt, State dict with Room_temp, Setpoint etc.
    """

    # Default tariff if not provided
    if tariff is None:
        tariff = {"hours": [0], "cost": [34]}

    # start_datetime = midnight of the given date
    start_datetime = datetime.combine(date, datetime.min.time())

    zone_schedule_key = f"zone_{zone}_heating_daily_schedule"

    optim_inputs = {
        "start_datetime":    start_datetime.isoformat(),
        "num_zones":         zone,
        zone_schedule_key:   schedule,
        "tariff":            tariff,
        f"zone_{zone}_temperature": zone_start_temp,
    }

    print(f"  Calling API for house {house_id} zone {zone} "
          f"on {date}...")
    print(f"  Inputs: {optim_inputs}")

    try:
        response = requests.post(
            api_url,
            json=optim_inputs,
            headers={"X-API-Key": API_KEY},   # ← correct header format
            timeout=30
        )
        response.raise_for_status()
        result = response.json()
        print(f"API call successful — "
              f"{len(result.get('real_dt', []))} timesteps returned")
        return result
    except requests.exceptions.RequestException as e:
        print(f"API call failed: {e}")
        return None

In [34]:
def build_node_from_api_response(
    call_id: str,
    house_id: int,
    zone: int,
    call_datetime: datetime,
    schedule: dict,
    tariff: dict,
    zone_start_temp: float,
    api_response: dict,
    overrides: list,
    real_outside_temp: float = None   # ← new parameter
) -> APICallNode:
    """
    Builds an APICallNode using real outside temperature from sensors

    """
    if not api_response.get("success", True):
        print(f"Warning: API returned success=False for {call_id}")

    state   = api_response.get("State", {})
    real_dt = api_response.get("real_dt", [])

    room_temp = state.get(f"room_temp.z{zone}", [])
    setpoint  = state.get(f"setpoint.z{zone}", [])
    power_in  = state.get(f"E.hs1.heat.z{zone}", [])
    power_out = state.get(f"U.hs1.z{zone}", [])
    ext_temps = state.get("ext", [])

    #  Use real outside temp if available, otherwise fall back to model 
    if real_outside_temp is not None and not pd.isna(real_outside_temp):
        outside_temp = float(real_outside_temp)
    else:
        outside_temp = float(np.mean([
            t for t in ext_temps if t is not None and isinstance(t, (int, float))
        ])) if ext_temps else 10.0

    # Override info
    override_happened = len(overrides) > 0
    override_time = overrides[0]["override_time"] if overrides else None
    override_temp = overrides[0]["override_temp"] if overrides else None

    node = APICallNode(
        call_id           = call_id,
        house_id          = house_id,
        zone              = zone,
        call_datetime     = call_datetime,
        outside_temp      = outside_temp,
        start_hour        = call_datetime.hour,
        schedule_hours    = schedule.get("hours", []),
        schedule_temps    = schedule.get("C", []),
        tariff_hours      = tariff.get("hours", []),
        tariff_costs      = tariff.get("cost", []),
        zone_start_temp   = zone_start_temp,
        setpoint          = setpoint,
        room_temp         = room_temp,
        power_in          = power_in,
        power_out         = power_out,
        real_dt           = real_dt,
        override_happened = override_happened,
        override_time     = override_time,
        override_temp     = override_temp
    )

    return node

In [35]:
def build_historical_nodes(
    merged_data: pd.DataFrame,
    new_data: pd.DataFrame,
    api_url: str,
    house_id: int,
    zone: int,
    tariff: dict = None
) -> list:
    """
    Builds the full list of APICallNodes using real room temperatures
    and real outside temperatures from the merged dataset.
    """
    df = new_data
    zone_mask = df["zone_int"].fillna(-1).astype(int) == int(zone)

    house_df = df[
        (df["house_id"] == house_id) &
        zone_mask
    ]

    unique_timestamps = sorted(house_df["Timestamp"].unique())

    print(f"\nBuilding historical nodes for house {house_id} zone {zone}...")
    print(f"Found {len(unique_timestamps)} unique event timestamps\n")

    all_nodes = []

    for event_ts in unique_timestamps:
        print(f"Processing event: {event_ts}")

        # Extract schedule and override as before
        schedule = extract_schedule_for_event(df, house_id, zone, event_ts)
        override = extract_override_for_event(df, house_id, zone, event_ts)

        if override:
            print(f"Override: {override['override_temp']}°C "
                  f"until {override['override_end']}")
        else:
            print(f"No override at this event")

        #  Look up real room temperature from merged data 
        event_row = merged_data[merged_data["Timestamp"] == event_ts]

        if not event_row.empty:
            if zone == 1:
                real_start_temp = event_row["Room temperature (Zone 1) (°C)"].values[0]
            else:
                real_start_temp = event_row["Room temperature (Zone 2) (°C)"].values[0]

            real_outside_temp = event_row["External temperature (°C)"].values[0]
        else:
            real_start_temp = None
            real_outside_temp = None

        if real_start_temp < 5.0:
            print(f"WARNING: Suspiciously low room temp ({real_start_temp}°C) — possible sensor fault")

        real_tariff = event_row["Tariff rate (p/kWh)"].values[0]
        if pd.isna(real_tariff) or real_tariff is None:
            real_tariff = 34.0
            print(f"No tariff reading — using default 34p/kWh")
        else:
            print(f"Real tariff from data: {real_tariff}p/kWh")

        tariff_dict = {"hours": [0], "cost": [float(real_tariff)]}

        # Fallback if sensor data is missing
        if pd.isna(real_start_temp) or real_start_temp is None:
            real_start_temp = 19.0
            print(f"No sensor reading — using default start temp 19.0°C")
        else:
            print(f"Real room temp from sensor: {real_start_temp:.1f}°C")

        if pd.isna(real_outside_temp) or real_outside_temp is None:
            real_outside_temp = 10.0
            print(f"No sensor reading — using default outside temp 10.0°C")
        else:
            print(f"Real outside temp from sensor: {real_outside_temp:.1f}°C")

        # ── Call API with real room temperature ──
        event_date = event_ts.date() if hasattr(event_ts, 'date') else \
                     pd.Timestamp(event_ts).date()

        api_response = call_api_for_day(
            api_url         = api_url,
            house_id        = house_id,
            zone            = zone,
            date            = event_date,
            schedule        = schedule,
            zone_start_temp = real_start_temp,   
            tariff          = tariff_dict
        )

        if api_response is None:
            print(f"Skipping — API call failed\n")
            continue

        # Build unique call_id
        ts_str = str(event_ts).replace(" ", "T").replace("+00:00", "Z")[:19]
        call_id = f"{ts_str}_{house_id}_z{zone}"

        overrides_list = [override] if override else []

        # ── Build node with real outside temperature ──
        node = build_node_from_api_response(
            call_id          = call_id,
            house_id         = house_id,
            zone             = zone,
            call_datetime    = pd.Timestamp(event_ts).to_pydatetime(),
            schedule         = schedule,
            tariff           = tariff_dict,
            zone_start_temp  = real_start_temp,
            api_response     = api_response,
            overrides        = overrides_list,
            real_outside_temp = real_outside_temp   # ← new parameter
        )

        all_nodes.append(node)
        print(f"Node built: {call_id} "
              f"(override={node.override_happened})\n")

    print(f"Total nodes built: {len(all_nodes)}")
    print(f"Nodes with overrides: "
          f"{sum(1 for n in all_nodes if n.override_happened)}")
    print(f"Nodes without overrides: "
          f"{sum(1 for n in all_nodes if not n.override_happened)}")

    return all_nodes

In [36]:
def run_real_example(new_data: pd.DataFrame, room_temp_df: pd.DataFrame):
    """
    End-to-end example 
 
    1. Prepares new_data ONCE (timezone, zone_int etc.)
    2. Builds historical nodes by calling the API for every day
    3. Finds the first override event
    4. Runs FACE to explain it
    5. Prints the plain English explanation
 
    INPUT: new_data DataFrame (already loaded in your notebook)
    """
 
    # Prepare data ONCE here — do not call prepare_new_data again 
    prepared_df = prepare_new_data(new_data)
 
    # Build the historical node log 
    all_nodes = build_historical_nodes(
        merged_data        = merged,
        new_data           = prepared_df,  
        api_url            = url,
        house_id           = HOUSE_ID,
        zone               = ZONE,
        tariff             = None
    )
 
    if not all_nodes:
        print("No nodes built — check API URL and new_data contents")
        return None, [], []
 
    # Find the first real override event 
    override_nodes = [n for n in all_nodes if n.override_happened]
 
    # Only keep overrides where user wanted it WARMER than it currently was
    meaningful_overrides = [
    n for n in override_nodes
    if n.override_temp > n.zone_start_temp
]
 
    print(f"Total overrides: {len(override_nodes)}")
    print(f"'Too cold' overrides (FACE can help): {len(meaningful_overrides)}")
    print(f"'Adjustment' overrides (already warm enough): {len(override_nodes) - len(meaningful_overrides)}")
 
    if not meaningful_overrides:
        print("No 'too cold' overrides found — the heating system is performing well.")
        return None, all_nodes, []
 
    for target_node in meaningful_overrides:
        result = run_face(
            api_url=url,
            all_nodes=all_nodes,
            override_node_id=target_node.call_id,
            override_time=target_node.override_time,
            override_temp=target_node.override_temp,
            zone=ZONE,
            room_temp_df=room_temp_df 
    )
    print(f"RUNNING FACE ON FIRST OVERRIDE EVENT")
    print(f"House: {target_node.house_id} | Zone: {target_node.zone}")
    print(f"Date: {target_node.call_datetime.date()}")
    print(f"Override time: {target_node.override_time}")
    print(f"User wanted: {target_node.override_temp}°C")
    print(f"Model had set: "
          f"{np.mean([t for t in target_node.schedule_temps if isinstance(t, (int, float))]):.1f}°C")
    print(f"Outside temp: {target_node.outside_temp:.1f}°C")
 
    sensor_temps = [n.zone_start_temp for n in all_nodes if n.zone_start_temp < 12.0]
    if len(sensor_temps) > len(all_nodes) * 0.3:
        print(f"WARNING: {len(sensor_temps)} out of {len(all_nodes)} nodes have room temps below 12°C.")
        print("Sensor data may be unreliable for this house. FACE results should be treated with caution.")
 
    # Run FACE 
    result = run_face(
        api_url          = url,
        all_nodes        = all_nodes,
        override_node_id = target_node.call_id,
        override_time    = target_node.override_time,
        override_temp    = target_node.override_temp,
        zone             = ZONE,
        room_temp_df     = room_temp_df,
        comfort_tolerance= 0.5
    )
 
    # Print results 

    print("FACE RESULT SUMMARY")

    print(f"Counterfactual found: {result['success']}")
    print(f"Steps in path: {len(result['cf_steps'])}")
    print(f"\nExplanation:")
    print(result["explanation"])
 
    #  Show step-by-step breakdown 
    if result["cf_steps"]:
        print("STEP BY STEP BREAKDOWN")
    
        for step in result["cf_steps"]:
            print(f"\nStep {step['step']}:")
            print(f"  Schedule hours: {step['schedule_hours']}")
            print(f"  Schedule temps: {step['schedule_temps']}")
            print(f"  Room temp achieved: {step['room_temp_achieved']}°C")
            print(f"  Comfort gap: {step['comfort_gap']}°C")
            print(f"  Override prevented: {step['override_prevented']}")
            print(f"  Extra cost: £{step['extra_cost_gbp']:.4f}")
 
    return result, all_nodes, meaningful_overrides


In [38]:
result, all_nodes, meaningful_overrides = run_real_example(new_data, room_temp_df)


Building historical nodes for house 29104 zone 1...
Found 32 unique event timestamps

Processing event: 2026-02-05 18:18:21+00:00
  Schedule at 2026-02-05 18:18:21+00:00: hours=[0, 6, 9, 16, 22], temps=[17.0, 19.0, 17.0, 20.0, 17.0]
No override at this event
No tariff reading — using default 34p/kWh
No sensor reading — using default start temp 19.0°C
No sensor reading — using default outside temp 10.0°C
  Calling API for house 29104 zone 1 on 2026-02-05...
  Inputs: {'start_datetime': '2026-02-05T00:00:00', 'num_zones': 1, 'zone_1_heating_daily_schedule': {'hours': [0, 6, 9, 16, 22], 'C': [17.0, 19.0, 17.0, 20.0, 17.0]}, 'tariff': {'hours': [0], 'cost': [34.0]}, 'zone_1_temperature': 19.0}
API call successful — 30 timesteps returned
Node built: 2026-02-05T18:18:21_29104_z1 (override=False)

Processing event: 2026-02-05 18:18:24+00:00
  Schedule at 2026-02-05 18:18:24+00:00: hours=[7, 17], temps=[19.0, 20.0]
No override at this event
No tariff reading — using default 34p/kWh
No sensor 

In [39]:
# Pick the first meaningful override that has sensor data
test_node = None
event_row = None

for node in meaningful_overrides:
    # Skip extreme overrides (above 23°C) — FACE can't help with these
    if node.override_temp > 23.0:
        continue
    row = merged[merged["Timestamp"] == pd.Timestamp(node.override_time)]
    if not row.empty:
        room_val = row["Room temperature (Zone 1) (°C)"].values[0]
        if pd.notna(room_val) and room_val > 0:
            test_node = node
            event_row = row
            break

if test_node is None:
    print("No suitable override found — trying all meaningful overrides:")
    for node in meaningful_overrides:
        print(f"  {node.call_id}: wanted {node.override_temp}°C, "
              f"room was {node.zone_start_temp}°C")
        
else:
    current_room_temp = float(event_row["Room temperature (Zone 1) (°C)"].values[0])
    outside_temp = float(event_row["External temperature (°C)"].values[0])
    tariff_rate = float(event_row["Tariff rate (p/kWh)"].values[0])

    # Handle NaN tariff
    if pd.isna(tariff_rate):
        tariff_rate = 34.0

    # Handle NaN outside temp
    if pd.isna(outside_temp):
        outside_temp = 10.0

    override_hour = test_node.override_time.hour
    target_hour = None
    for h in test_node.schedule_hours:
        if h > override_hour:
            target_hour = h
            break
    if target_hour is None:
        target_hour = min(override_hour + 1, 23)

    target_time = test_node.override_time.replace(
        hour=target_hour, minute=0, second=0, tzinfo=None
    )
    current_time = test_node.override_time.replace(tzinfo=None)

    print(f"Testing override from {test_node.call_id}")
    print(f"Room temp: {current_room_temp}°C")
    print(f"Outside: {outside_temp}°C")
    print(f"User wants: {test_node.override_temp}°C")
    print(f"Tariff: {tariff_rate}p/kWh")
    print(f"Schedule target time: {target_time}")
    print()

    rt_result = real_time_override_check_with_face(
        api_url=url,
        zone=ZONE,
        current_room_temp=current_room_temp,
        outside_temp=outside_temp,
        current_schedule={
            "hours": test_node.schedule_hours,
            "C": test_node.schedule_temps
        },
        current_tariff={"hours": [0], "cost": [tariff_rate]},
        override_temp=test_node.override_temp,
        target_time=target_time,
        current_time=current_time,
        all_nodes=all_nodes,
        room_temp_df=room_temp_df
    )

    if rt_result.get("success"):
        print("SUMMARY")
        print(f"Recommendation: {rt_result['recommendation']}")
        print(f"Predicted temp if wait: {rt_result['predicted_temp_if_wait']}°C")
        print(f"Baseline cost: £{rt_result['baseline_cost_gbp']}")
        print(f"Override cost: £{rt_result['override_cost_gbp']}")
        print(f"Extra cost: £{rt_result['extra_cost_gbp']}")
        print(f"Similar past overrides: {rt_result['similar_past_overrides']}")
        print(f"FACE found solution: {rt_result.get('face_found_solution', False)}")
    else:
        print(f"Check failed: {rt_result.get('message', 'Unknown error')}")

Testing override from 2026-02-07T00:08:30_29104_z1
Room temp: 20.9°C
Outside: 7.0°C
User wants: 21.5°C
Tariff: 28.82943p/kWh
Schedule target time: 2026-02-07 08:00:00

REAL-TIME OVERRIDE CHECK
Room: 20.9°C | Outside: 7.0°C
User wants: 21.5°C | Schedule target time: 2026-02-07 08:00:00
Temperature gap: 0.6°C
Hours until target: 7.9
Estimated override cost: £3.32
Estimated baseline cost: £3.15
Extra cost: £0.17
Predicted temp if wait: 22.0°C
FACE COUNTERFACTUAL EXPLANATION
Override detected: zone 1, time 2026-02-07 00:08:30+00:00, user wanted 21.5°C
Found 4 target nodes (days where 21.5°C was reached without override)
Building FACE graph from 32 historical API calls...
Graph built: 32 nodes, 238 edges

Trying path to 2026-02-06T01:22:58_29104_z1 (2 steps)
Path: 2026-02-07T00:08:30_29104_z12026-02-06T01:22:58_29104_z1

Validating CF step 1/1: node 2026-02-06T01:22:58_29104_z1
REAL room temp at 00:08 on 2026-02-06: 21.6°C (wanted 21.5°C, gap=-0.1°C)
Override prevented: True
 CF found at st